# 階段九：提取技能需求 (job_skill_requirement)

依 **`cleaner步驟_v2.md`** 階段九，從職缺提取技能並寫入 `job_skill_requirement`。技能來源有兩種：
1. **結構化欄位**：`skills`、`tools`（逗號/頓號拆分）
2. **JD/Requirements 文字**：依 **Supabase 欄位** `job_description`、`requirements` 做關鍵字整詞匹配（若資料來源為 104 原始 CSV，可能為 `other_requirements`），補上 skills/tools 為空或不足的職缺。

**執行方式**：由上而下一個 cell 一個 cell 執行，每步可檢查輸出以確認資料邏輯正確。

**前置條件**：
- `skill_master` 已建立並有核心技能（可先跑 `skill_write_evaluation.ipynb` 擴充）
- `job_posting`、`company_info` 已寫入 Supabase
- 工作目錄為 `supabase_control`，具備 `jobs_rows.csv` 或 `jobs_cleaned.csv`

---
## 環境設定與連線

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()
if not os.environ.get("SUPABASE_URL"):
    load_dotenv(os.path.join(os.getcwd(), ".env"))

SUPABASE_URL = os.environ.get("SUPABASE_URL")
SUPABASE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY") or os.environ.get("SUPABASE_KEY")
assert SUPABASE_URL and SUPABASE_KEY, "請設定 SUPABASE_URL 與 SUPABASE_KEY（.env）"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
DATA_DIR = os.getcwd()

# 資料來源：jobs_rows.csv（原始）或 jobs_cleaned.csv（階段八匯出）。若用 jobs_rows，比對時建議做簡單 strip 以利對上 DB 的 job_title。
RAW_CSV = os.path.join(DATA_DIR, "jobs_rows.csv")
if not os.path.isfile(RAW_CSV):
    RAW_CSV = os.path.join(DATA_DIR, "jobs_cleaned.csv")
print(f"使用資料檔：{RAW_CSV}")
print(f"Supabase 連線：{SUPABASE_URL[:50]}...")

使用資料檔：c:\Users\Elvis\git\final\supabase_control\jobs_rows.csv
Supabase 連線：https://nyslsqlgsavvfwiducdu.supabase.co...


---
## 本階段九 步驟 1：建立技能映射表

從 Supabase `skill_master` 讀取 `skill_id, skill_name, synonyms`，建立「技能名稱（含同義詞）→ skill_id」的查找表，供後續精確匹配。

In [2]:
# 從 Supabase 讀取 skill_master（若筆數很多可改為分頁，這裡先 limit 5000）
response = supabase.table("skill_master").select("skill_id, skill_name, synonyms").limit(5000).execute()
skill_master_df = pd.DataFrame(response.data or [])

print(f"✅ 讀取了 {len(skill_master_df)} 個技能")
if len(skill_master_df) == 0:
    raise RuntimeError("skill_master 為空，請先填入技能或執行 skill_write_evaluation.ipynb")

# 建立同義詞反向索引（標準名稱 + 所有同義詞 → skill_id）
synonym_to_skill_id = {}

for _, row in skill_master_df.iterrows():
    skill_id = row["skill_id"]
    skill_name = row["skill_name"]
    synonyms_raw = row.get("synonyms")
    
    if synonyms_raw is not None:
        if isinstance(synonyms_raw, str):
            try:
                synonyms = json.loads(synonyms_raw)
            except Exception:
                synonyms = []
        else:
            synonyms = list(synonyms_raw) if synonyms_raw else []
    else:
        synonyms = []
    
    synonym_to_skill_id[str(skill_name).strip().lower()] = skill_id
    for syn in synonyms:
        if syn is not None and str(syn).strip():
            synonym_to_skill_id[str(syn).strip().lower()] = skill_id

# 供 JD/Requirements 關鍵字匹配用：skill_id -> [(compiled_regex, phrase), ...] 依 phrase 長度由長到短
import re
skill_id_to_jd_patterns = {}
for _, row in skill_master_df.iterrows():
    skill_id = row["skill_id"]
    skill_name = str(row["skill_name"]).strip()
    synonyms_raw = row.get("synonyms")
    synonyms = []
    if synonyms_raw is not None:
        if isinstance(synonyms_raw, str):
            try:
                synonyms = json.loads(synonyms_raw)
            except Exception:
                pass
        else:
            synonyms = list(synonyms_raw) if synonyms_raw else []
    phrases = [skill_name] + [str(s).strip() for s in synonyms if s and len(str(s).strip()) >= 2]
    phrases = list(dict.fromkeys(p for p in phrases if len(p) >= 2))
    if not phrases:
        continue
    # 長 phrase 先匹配，避免 "Spring" 吃掉 "Spring Boot"
    patterns = []
    for p in sorted(phrases, key=len, reverse=True):
        try:
            pat = re.compile(r"(?<![a-zA-Z0-9_])" + re.escape(p) + r"(?![a-zA-Z0-9_])", re.IGNORECASE)
            patterns.append((pat, p))
        except Exception:
            continue
    if patterns:
        skill_id_to_jd_patterns[skill_id] = patterns

print(f"✅ 建立了 {len(synonym_to_skill_id)} 個技能映射（含同義詞）")
print(f"✅ JD 關鍵字匹配：{len(skill_id_to_jd_patterns)} 個技能有整詞 regex")
display(skill_master_df.head(10))

✅ 讀取了 70 個技能
✅ 建立了 135 個技能映射（含同義詞）
✅ JD 關鍵字匹配：70 個技能有整詞 regex


,skill_id,skill_name,synonyms
0,1,Python,"[python, py]"
1,2,JavaScript,"[JS, js, javascript, ECMAScript]"
2,3,Java,[java]
3,4,TypeScript,"[TS, ts, typescript]"
4,5,C++,"[cpp, c plus plus]"
5,6,Go,"[Golang, golang]"
6,7,Ruby,[ruby]
7,8,PHP,[php]
8,9,Swift,[swift]
9,10,Kotlin,[kotlin]


---
## 本階段九 步驟 2：從 raw_data 提取技能

1. **結構化**：合併 `skills`、`tools` 並拆成技能列表（支援逗號、全形頓號）。
2. **JD/Requirements 文字**：合併 **Supabase 對應欄位** `job_description`、`requirements` 成一段文字（若讀取 104 原始 CSV 則可能為 `other_requirements`），供後續關鍵字整詞匹配，補足 skills/tools 為空的職缺。

In [3]:
# 讀取原始職缺資料
raw_df = pd.read_csv(RAW_CSV, encoding="utf-8", low_memory=False)
print(f"✅ 讀取了 {len(raw_df)} 筆職缺")
print(f"欄位：{list(raw_df.columns)}")

# 必要欄位：至少要有 company_name、職稱、以及「結構化技能」或「JD 文字」其一
has_company = "company_name" in raw_df.columns
has_job = "job_title" in raw_df.columns or "job_name" in raw_df.columns
has_structured = ("skills" in raw_df.columns or "tools" in raw_df.columns) 
# Supabase 欄位為 job_description / requirements；104 原始 CSV 可能為 other_requirements
has_jd_text = ("job_description" in raw_df.columns or "requirements" in raw_df.columns or "other_requirements" in raw_df.columns)
if not (has_company and has_job and (has_structured or has_jd_text)):
    raise ValueError("缺少必要欄位：需 company_name、job_title/job_name、以及 (skills/tools) 或 (job_description/requirements)")

✅ 讀取了 18729 筆職缺
欄位：['job_id', 'job_name', 'company_name', 'company_url', 'update_date', 'url', 'status', 'job_description', 'job_category', 'salary', 'job_type', 'location', 'management', 'business_trip', 'work_time', 'vacation', 'start_work', 'headcount', 'work_exp', 'education', 'major', 'language', 'skills', 'tools', 'certificates', 'other_requirements', 'legal_benefits', 'other_benefits', 'raw_benefits', 'contact_info', 'actively_hiring', 'applicants', 'created_at']


In [4]:
def parse_skills(skills_str, tools_str):
    """合併 skills 與 tools，拆成技能列表。支援逗號、全形頓號。"""
    all_skills = []
    for raw in (skills_str, tools_str):
        if pd.isna(raw) or not str(raw).strip():
            continue
        s = str(raw).strip()
        for sep in ["、", ","]:
            s = s.replace(sep, "|")
        parts = [x.strip() for x in s.split("|") if x.strip()]
        all_skills.extend(parts)
    return list(dict.fromkeys(all_skills))

raw_df["parsed_skills"] = raw_df.apply(
    lambda row: parse_skills(row.get("skills"), row.get("tools")),
    axis=1
)

# 合併 JD/Requirements 文字（對應 Supabase 欄位：job_description、requirements；104 CSV 可能為 other_requirements）
def build_jd_text(row):
    parts = []
    for col in ("job_description", "requirements", "other_requirements"):
        if col in row.index and pd.notna(row.get(col)) and str(row.get(col)).strip():
            parts.append(str(row[col]).strip())
    return " ".join(parts) if parts else ""

raw_df["jd_requirements_text"] = raw_df.apply(build_jd_text, axis=1)
n_with_jd = raw_df["jd_requirements_text"].str.len().gt(0).sum()
n_empty_structured = raw_df["parsed_skills"].apply(len).eq(0).sum()
print(f"✅ 解析了 {len(raw_df)} 筆職缺的結構化技能 (skills+tools)")
print(f"📊 有 JD/Requirements 文字的筆數：{n_with_jd}；結構化為空的筆數：{n_empty_structured}")
total_skills = sum(len(s) for s in raw_df["parsed_skills"])
avg_skills = total_skills / len(raw_df) if len(raw_df) else 0
print(f"📊 結構化技能總數（含重複）：{total_skills}，平均每筆：{avg_skills:.1f} 個")
raw_df[["company_name", "job_title" if "job_title" in raw_df.columns else "job_name", "parsed_skills", "jd_requirements_text"]].assign(jd_len=raw_df["jd_requirements_text"].str.len()).head(5)

✅ 解析了 18729 筆職缺的結構化技能 (skills+tools)
📊 有 JD/Requirements 文字的筆數：18107；結構化為空的筆數：5025
📊 結構化技能總數（含重複）：102102，平均每筆：5.5 個


,company_name,job_name,parsed_skills,jd_requirements_text,jd_len
0,泛太資訊科技開發股份有限公司,C#軟體工程師,"[軟體程式設計, 資料庫程式設計, Windows 2003, Windows XP, AS...",- 熟悉C#、.NET 、ASP.Net\n-依據系統需求規格進行程式設計與開發\n-使用 ...,150
1,Inventec_英業達股份有限公司,[NB]軟體品保工程師(士林),[],1.軟體測試規劃、建立測試環境與執行產品測試。\n2.負責產品相容性及整合測試。\n3.協助...,93
2,Allion_百佳泰股份有限公司,測試人員-實習/工讀 (中興新村南投高等研究園區),[作業系統基本操作],1. 電腦整合測試。\n2. 系統維護操作。\n3. 歡迎對電子產品有興趣者。 1. 大學...,118
3,億光電子工業股份有限公司,網管與雲端專員,"[Microsoft Azure, AWS, Security, Excel, PowerP...",1.企業網路問題的排除。\n2.資訊機房的維護與備份系統管理。\n3.Windows AD管...,312
4,DC數位溝通整合行銷_傳旭科技股份有限公司,專案管理師｜AI技能 + 專案歷練一次到位,[],想要的不只是工作，而是能讓你持續進步的舞台？\n\n在傳旭，我們讓專案管理不只停留在「控管進...,1189


---
## 本階段九 步驟 3：匹配技能並建立關聯記錄

1. 從 Supabase 讀取 `job_posting` 與 `company_info`，建立 `(company_name, job_title) → job_id` 映射。  
2. 對每筆職缺：  
   - **結構化**：`parsed_skills` 用 `synonym_to_skill_id` 匹配得到一批 `skill_id`。  
   - **JD/Requirements**：在 `jd_requirements_text` 上用 `skill_id_to_jd_patterns` 做整詞匹配，再補上未出現在結構化裡的 `skill_id`。  
3. 合併兩來源、去重後產出 `job_skill_records` 與 `unmatched_skills`（僅結構化欄位中未匹配的詞）。  
4. **importance / proficiency_level**：依該技能在 JD 中的**前後文**推論。importance 用「加分/佳/優先」→ nice-to-have、「必備/必要/須具備」→ required；proficiency 用「精通」→8、「熟悉/熟練」→6、「N 年」→3～7 等。若技能未出現在 JD 文字中則維持 `required` 與 `None`。  

**注意**：DB 的 `job_title` 來自清理後的職稱；若使用 `jobs_rows.csv`，比對鍵用 `job_name`，若公司名/職稱在清理時被改過，可能對不上，可改用手動清理過的 `jobs_cleaned.csv` 或對 raw 的 company_name / job_name 做相同清理再比對。

In [5]:
# 讀取 job_posting 與 company_info，建立 (company_name, job_title) -> job_id
job_response = supabase.table("job_posting").select("job_id, company_id, job_title").limit(20000).execute()
company_response = supabase.table("company_info").select("company_id, company_name").limit(10000).execute()

job_posting_df = pd.DataFrame(job_response.data or [])
company_df = pd.DataFrame(company_response.data or [])
job_with_company = job_posting_df.merge(company_df, on="company_id")

job_mapping = {}
for _, row in job_with_company.iterrows():
    key = (str(row["company_name"]).strip(), str(row["job_title"]).strip())
    job_mapping[key] = row["job_id"]

print(f"✅ 建立了 {len(job_mapping)} 個職缺映射 (company_name, job_title) -> job_id")
print("範例鍵（前 3 個）：", list(job_mapping.keys())[:3])

✅ 建立了 16070 個職缺映射 (company_name, job_title) -> job_id
範例鍵（前 3 個）： [('原誠有限公司', '【PHP後端工程師 (資深) ☆遠端高彈性辦公！優秀者可議☆】'), ('巨量移動科技股份有限公司', '資料科學家實習生-有機會派駐美西'), ('冠宇國際電子股份有限公司', '系統軟體設計工程師')]


In [6]:
# 比對時使用的職稱欄位：jobs_cleaned 用 job_title，jobs_rows 用 job_name
job_col = "job_title" if "job_title" in raw_df.columns else "job_name"


def infer_importance_and_proficiency(text, skill_id, skill_id_to_jd_patterns, before=100, after=80):
    """從 JD 文字中該技能出現處的前後文推論 importance 與 proficiency_level。"""
    importance = "required"
    proficiency_level = None
    if not text or not skill_id_to_jd_patterns or skill_id not in skill_id_to_jd_patterns:
        return importance, proficiency_level
    context = ""
    for pat, _ in skill_id_to_jd_patterns[skill_id]:
        m = pat.search(text)
        if m:
            start = max(0, m.start() - before)
            end = min(len(text), m.end() + after)
            context = text[start:end]
            break
    if not context:
        return importance, proficiency_level
    # importance：加分/佳/優先 → nice-to-have；必備/必要/須具備 → required
    if re.search(r"加分|佳|優先考慮|preferred|nice\s*to\s*have", context, re.I):
        importance = "nice-to-have"
    elif re.search(r"必備|必要條件|須具備|required", context, re.I):
        importance = "required"
    # proficiency_level：精通→8, 熟悉/熟練→6, 具備/了解→4, 初學→2；N年→依年數
    if re.search(r"精通", context):
        proficiency_level = 8
    elif re.search(r"熟悉|熟練", context):
        proficiency_level = 6
    elif re.search(r"具備|了解", context):
        proficiency_level = 4
    elif re.search(r"初學", context):
        proficiency_level = 2
    else:
        ym = re.search(r"(\d+)\s*年", context)
        if ym:
            y = int(ym.group(1))
            proficiency_level = 7 if y >= 5 else (5 if y >= 3 else (4 if y >= 2 else 3))
    return importance, proficiency_level


job_skill_records = []
unmatched_skills = set()
unmatched_jobs = 0
jd_added_count = 0  # 僅由 JD 關鍵字補上的 (job_id, skill_id) 筆數

for _, row in raw_df.iterrows():
    company_name = str(row["company_name"]).strip() if pd.notna(row.get("company_name")) else ""
    job_title = str(row[job_col]).strip() if pd.notna(row.get(job_col)) else ""
    key = (company_name, job_title)
    job_id = job_mapping.get(key)
    
    if not job_id:
        unmatched_jobs += 1
        continue
    
    # 來源 1：結構化 skills/tools
    skill_ids_from_structured = set()
    for skill_name in row["parsed_skills"]:
        if not skill_name or len(str(skill_name).strip()) < 2:
            continue
        sk = str(skill_name).strip().lower()
        skill_id = synonym_to_skill_id.get(sk)
        if skill_id is not None:
            skill_ids_from_structured.add(skill_id)
        else:
            unmatched_skills.add(skill_name)
    
    # 來源 2：JD/Requirements 關鍵字整詞匹配（只補尚未有的 skill_id）
    text = row.get("jd_requirements_text") or ""
    skill_ids_from_jd = set()
    if text and skill_id_to_jd_patterns:
        for sid, pattern_list in skill_id_to_jd_patterns.items():
            if sid in skill_ids_from_structured:
                continue
            for pat, _ in pattern_list:
                if pat.search(text):
                    skill_ids_from_jd.add(sid)
                    break
    
    # 合併兩來源，依 JD 前後文推論 importance / proficiency_level 後寫入
    all_skill_ids = skill_ids_from_structured | skill_ids_from_jd
    jd_added_count += len(skill_ids_from_jd)
    for skill_id in all_skill_ids:
        imp, prof = infer_importance_and_proficiency(text, skill_id, skill_id_to_jd_patterns)
        job_skill_records.append({
            "job_id": job_id,
            "skill_id": skill_id,
            "importance": imp,
            "proficiency_level": prof
        })

print(f"✅ 成功匹配 {len(job_skill_records)} 筆技能需求（含結構化 + JD 關鍵字）")
print(f"📊 其中僅由 JD/Requirements 關鍵字補上的 skill 關聯數：{jd_added_count}")
print(f"⚠️ 未匹配到 job_id 的職缺數：{unmatched_jobs}")
print(f"⚠️ 結構化欄位中未匹配技能種類數：{len(unmatched_skills)}")
if unmatched_skills:
    print(f"未匹配技能範例（前 15 個）：{sorted(unmatched_skills)[:15]}")

✅ 成功匹配 86470 筆技能需求（含結構化 + JD 關鍵字）
📊 其中僅由 JD/Requirements 關鍵字補上的 skill 關聯數：47497
⚠️ 未匹配到 job_id 的職缺數：913
⚠️ 結構化欄位中未匹配技能種類數：1216
未匹配技能範例（前 15 個）：['3ds Max', '3ds Max Design', '6 Sigma', 'ABAP', 'ABAQUS', 'ACL', 'ACT', 'ADC', 'ADO', 'ADSL', 'AI', 'AIX', 'AIx', 'AJAX', 'ANSI SQL']


---
## 步驟 3.5：更新既有資料的 importance / proficiency_level（可選）

若 **job_skill_requirement 已寫入過**，可用上一步產出的 `job_skill_records`（含推論後的 importance / proficiency_level）**僅 UPDATE 既有列**，不重複 INSERT。請先跑完步驟 1～3 再執行下方 cell。

In [7]:
# 僅 UPDATE 既有列：以 (job_id, skill_id) 匹配，更新 importance 與 proficiency_level
# 使用 .select() 回傳符合的列，依 response 筆數判斷是否真的更新到 DB
if not job_skill_records:
    print("請先執行上方步驟 3，產生 job_skill_records 後再執行本 cell。")
else:
    df_update = pd.DataFrame(job_skill_records).drop_duplicates(subset=["job_id", "skill_id"])
    total = len(df_update)
    updated = 0
    no_match = 0
    errors = []
    BATCH_SIZE = 500
    for i in range(0, total, BATCH_SIZE):
        batch = df_update.iloc[i : i + BATCH_SIZE]
        for _, row in batch.iterrows():
            jid, sid = int(row["job_id"]), int(row["skill_id"])
            imp = str(row["importance"]) if pd.notna(row["importance"]) else "required"
            prof = row["proficiency_level"] if pd.notna(row["proficiency_level"]) else None
            if prof is not None:
                prof = int(prof)
            try:
                resp = supabase.table("job_skill_requirement").update({
                    "importance": imp,
                    "proficiency_level": prof
                }).eq("job_id", jid).eq("skill_id", sid).select("requirement_id").execute()
                if resp.data and len(resp.data) > 0:
                    updated += 1
                else:
                    no_match += 1
            except Exception as e:
                errors.append((jid, sid, str(e)))
        print(f"已處理 {min(i + BATCH_SIZE, total)} / {total} 筆（實際更新 {updated}，無匹配 {no_match}）")
    print(f"✅ 完成：實際更新 {updated} 筆；無匹配 (job_id, skill_id) {no_match} 筆")
    if errors:
        print(f"⚠️ 例外 {len(errors)} 筆：{errors[:3]}...")

已處理 500 / 72373 筆（實際更新 0，無匹配 0）
已處理 1000 / 72373 筆（實際更新 0，無匹配 0）
已處理 1500 / 72373 筆（實際更新 0，無匹配 0）
已處理 2000 / 72373 筆（實際更新 0，無匹配 0）
已處理 2500 / 72373 筆（實際更新 0，無匹配 0）
已處理 3000 / 72373 筆（實際更新 0，無匹配 0）
已處理 3500 / 72373 筆（實際更新 0，無匹配 0）
已處理 4000 / 72373 筆（實際更新 0，無匹配 0）
已處理 4500 / 72373 筆（實際更新 0，無匹配 0）
已處理 5000 / 72373 筆（實際更新 0，無匹配 0）
已處理 5500 / 72373 筆（實際更新 0，無匹配 0）
已處理 6000 / 72373 筆（實際更新 0，無匹配 0）
已處理 6500 / 72373 筆（實際更新 0，無匹配 0）
已處理 7000 / 72373 筆（實際更新 0，無匹配 0）
已處理 7500 / 72373 筆（實際更新 0，無匹配 0）
已處理 8000 / 72373 筆（實際更新 0，無匹配 0）
已處理 8500 / 72373 筆（實際更新 0，無匹配 0）
已處理 9000 / 72373 筆（實際更新 0，無匹配 0）
已處理 9500 / 72373 筆（實際更新 0，無匹配 0）
已處理 10000 / 72373 筆（實際更新 0，無匹配 0）
已處理 10500 / 72373 筆（實際更新 0，無匹配 0）
已處理 11000 / 72373 筆（實際更新 0，無匹配 0）
已處理 11500 / 72373 筆（實際更新 0，無匹配 0）
已處理 12000 / 72373 筆（實際更新 0，無匹配 0）
已處理 12500 / 72373 筆（實際更新 0，無匹配 0）
已處理 13000 / 72373 筆（實際更新 0，無匹配 0）
已處理 13500 / 72373 筆（實際更新 0，無匹配 0）
已處理 14000 / 72373 筆（實際更新 0，無匹配 0）
已處理 14500 / 72373 筆（實際更新 0，無匹配 0）
已處理 15000 / 72373 筆（實際更新 0，無匹配 0）


**步驟 3.5 驗證**：跑完上方 UPDATE 後可執行下方 cell，查 DB 中 `importance` / `proficiency_level` 的分布，確認是否有寫入。

In [8]:
# 查 DB：importance / proficiency_level 分布（確認 UPDATE 是否生效）
r = supabase.table("job_skill_requirement").select("importance", "proficiency_level").limit(50000).execute()
df_check = pd.DataFrame(r.data)
print("importance 分布："); print(df_check["importance"].value_counts(dropna=False))
print("\nproficiency_level 非空筆數：", df_check["proficiency_level"].notna().sum())
print("前 5 筆（含 proficiency_level）："); display(df_check.head())

importance 分布：
importance
required    50000
Name: count, dtype: int64

proficiency_level 非空筆數： 0
前 5 筆（含 proficiency_level）：


,importance,proficiency_level
0,required,None
1,required,None
2,required,None
3,required,None
4,required,None


---
## 本階段九 步驟 4：去重並寫入 job_skill_requirement

同一職缺不重複同一技能；先預覽去重後筆數，再決定是否寫入。

In [11]:
job_skill_df = pd.DataFrame(job_skill_records)

if len(job_skill_df) == 0:
    print("❌ 沒有可寫入的技能需求記錄，請檢查步驟 1～3 的資料與映射")
else:
    before = len(job_skill_df)
    job_skill_df = job_skill_df.drop_duplicates(subset=["job_id", "skill_id"])
    after = len(job_skill_df)
    print(f"去重前：{before} 筆，去重後：{after} 筆")
    display(job_skill_df.head(10))
    
    # 寫入開關：設為 True 才會真的寫入 Supabase
    DO_INSERT = True
    if DO_INSERT:
        BATCH_SIZE = 500
        total_inserted = 0
        errors = []
        for i in range(0, len(job_skill_df), BATCH_SIZE):
            batch = job_skill_df.iloc[i:i+BATCH_SIZE].to_dict("records")
            try:
                supabase.table("job_skill_requirement").insert(batch).execute()
                total_inserted += len(batch)
                print(f"寫入 {total_inserted}/{len(job_skill_df)}")
            except Exception as e:
                errors.append((i, str(e)))
                print(f"❌ 批次 {i} 失敗：{e}")
        print(f"✅ 實際寫入 {total_inserted} 筆")
        if errors:
            print(f"失敗批次數：{len(errors)}")
    else:
        print("（未寫入）請確認資料無誤後，將 DO_INSERT 改為 True 再執行此 cell")

去重前：86470 筆，去重後：72373 筆


,job_id,skill_id,importance,proficiency_level
0,1,65,required,None
1,1,2,required,None
2,1,41,required,None
3,1,42,required,None
4,1,47,required,None
5,20134,24,required,None
6,20134,25,required,None
7,20134,43,required,None
8,15656,38,required,None
9,15656,39,required,None


寫入 500/72373
寫入 1000/72373
寫入 1500/72373
寫入 2000/72373
寫入 2500/72373
寫入 3000/72373
寫入 3500/72373
寫入 4000/72373
寫入 4500/72373
寫入 5000/72373
寫入 5500/72373
寫入 6000/72373
寫入 6500/72373
寫入 7000/72373
寫入 7500/72373
寫入 8000/72373
寫入 8500/72373
寫入 9000/72373
寫入 9500/72373
寫入 10000/72373
寫入 10500/72373
寫入 11000/72373
寫入 11500/72373
寫入 12000/72373
寫入 12500/72373
寫入 13000/72373
寫入 13500/72373
寫入 14000/72373
寫入 14500/72373
寫入 15000/72373
寫入 15500/72373
寫入 16000/72373
寫入 16500/72373
寫入 17000/72373
寫入 17500/72373
寫入 18000/72373
寫入 18500/72373
寫入 19000/72373
寫入 19500/72373
寫入 20000/72373
寫入 20500/72373
寫入 21000/72373
寫入 21500/72373
寫入 22000/72373
寫入 22500/72373
寫入 23000/72373
寫入 23500/72373
寫入 24000/72373
寫入 24500/72373
寫入 25000/72373
寫入 25500/72373
寫入 26000/72373
寫入 26500/72373
寫入 27000/72373
寫入 27500/72373
寫入 28000/72373
寫入 28500/72373
寫入 29000/72373
寫入 29500/72373
寫入 30000/72373
寫入 30500/72373
寫入 31000/72373
寫入 31500/72373
寫入 32000/72373
寫入 32500/72373
寫入 33000/72373
寫入 33500/72373
寫入 34000/72373


---
## 本階段九 步驟 5：匯出未匹配技能

將未匹配技能寫入 `unmatched_skills.csv`，供手動分類或交給 `skill_write_evaluation.ipynb` 補 skill_master 後重跑本階段九步驟 1～4。

In [8]:
if unmatched_skills:
    unmatched_df = pd.DataFrame({
        "skill_name": sorted(unmatched_skills),
        "skill_category": None,
        "synonyms": None,
        "notes": ""
    })
    out_path = os.path.join(DATA_DIR, "unmatched_skills.csv")
    unmatched_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"✅ 已匯出 {len(unmatched_skills)} 個未匹配技能到 {out_path}")
else:
    print("✅ 沒有未匹配技能，無需匯出")

✅ 已匯出 1216 個未匹配技能到 c:\Users\Elvis\git\final\supabase_control\unmatched_skills.csv


---
## 本階段九 步驟 6：驗證寫入結果

若已寫入，可用下列查詢檢查總數與分布。

In [12]:
resp = supabase.table("job_skill_requirement").select("*", count="exact").limit(1).execute()
total = getattr(resp, "count", None) or 0
print(f"✅ job_skill_requirement 總筆數：{total}")
if total > 0:
    sample = supabase.table("job_skill_requirement").select("job_id, skill_id, importance").limit(5).execute()
    print("範例：")
    display(pd.DataFrame(sample.data))

✅ job_skill_requirement 總筆數：72373
範例：


,job_id,skill_id,importance
0,1,65,required
1,1,2,required
2,1,41,required
3,1,42,required
4,1,47,required
